### 🔗 Links and Resources
- [pivot()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.GroupedData.pivot.html)
- [unpivot()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.unpivot.html#pyspark.sql.DataFrame.unpivot)

In [0]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel — 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()


def display(df):
    df.show(truncate=False)

### 📌 Reading data into a Spark DataFrame

In [0]:
df = spark.read.table("population_metrics.default.countries_consolidated")

df.show(truncate=False)

### 📌 pivot()

In [0]:
# Aggregate population by sub-region into separate region columns
df_pivot = (
    df
    .groupBy("sub_region")   # group rows by each sub-region
    .pivot("region")         # turn each distinct region value into its own column
    .sum("population")       # compute the sum of population for each (sub_region, region) pair
)

df_pivot.show(truncate=False)

### 📌 unpivot()

In [0]:
# Take the df_pivot DataFrame and “melt” the specified continent columns
df_pivot.unpivot(
    "sub_region",                                            # column holding the IDs (grouping key)
    ["Africa", "America", "Asia", "Europe", "Oceania"],      # list of columns to unpivot (these become row values)
    "region",                                                # name for the new column that will hold the former column names
    "population"                                             # name for the new column that will hold the corresponding values
).show(truncate=False)                              